# NoisiQ: Tangible Quantum Circuit Noise

NoisiQ is a Python library designed to make quantum circuit noise tangible. It allows you to build circuits, attach realistic noise models, and observe how errors propagate through the circuit using step-by-step animations and heatmaps.

This notebook demonstrates the core capabilities of the NoisiQ MVP:
1. **High-Performance Backend**: Using `Stim` for polynomial-time Clifford simulation.
2. **Layer-by-Layer Tracking**: Explicitly sampling and recording where errors originate.
3. **Interactive Visualization**: Stepping through error propagation with `ipywidgets` and `matplotlib`.
4. **Error Heatmaps**: Visualizing error severity on top of Qiskit-style circuit diagrams.

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
from noisiq.ir import Circuit, gates
from noisiq.noise import depolarizing_error, bit_flip_error
from noisiq.visualization import Visualizer, draw_error_heatmap

## 1. Building a Circuit

We'll start by building a simple Bell state circuit ($|00\rangle + |11\rangle$).

In [ ]:
c = Circuit(n_qubits=2, name="Bell State")
c.add_gate(gates.H, (0,))
c.add_gate(gates.CNOT, (0, 1))

print(c)

## 2. Configuring Noise

We can attach noise models to specific gates in the circuit. Here, we'll add a 10% depolarizing error to the CNOT gate (index 1).

In [ ]:
# Noise config maps gate_index to a noise model
noise_config = {
    1: depolarizing_error(0.1)
}

print(f"Noise at CNOT: {noise_config[1]}")

## 3. Interactive Visualization

The `Visualizer` class runs the simulation and provides an interactive slider to step through the execution. 

**Note**: In the visualization, gates that trigger a stochastic error will be highlighted in **red**. The first render might take a second as it initializes the Qiskit drawer.

In [ ]:
vis = Visualizer(c)
vis.simulate(noise_config=noise_config, seed=42)
vis.show()

## 4. Static Heatmap

We can also generate a static heatmap showing the aggregate errors (or a specific snapshot) without the interactive slider.

In [ ]:
# Show the circuit state after the final step
fig = draw_error_heatmap(c, vis.result, step_index=1)
plt.title("Error Origin at CNOT Layer")
plt.show()

## 5. Complex Example: 5-Qubit GHZ State

Let's see how errors propagate in a larger circuit.

In [ ]:
n = 5
ghz = Circuit(n_qubits=n, name="5-Qubit GHZ")
ghz.add_gate(gates.H, (0,))
for i in range(n - 1):
    ghz.add_gate(gates.CNOT, (i, i + 1))

# Add bit-flip noise to the first few CNOTs
ghz_noise = {
    1: bit_flip_error(0.5),
    2: bit_flip_error(0.5)
}

vis_ghz = Visualizer(ghz)
vis_ghz.simulate(noise_config=ghz_noise, seed=123)
vis_ghz.show()